In [1]:
!pip install ultralytics opencv-python matplotlib pycocotools pyyamlpip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install tensorflow


Looking in indexes: https://download.pytorch.org/whl/cu121


ERROR: Could not find a version that satisfies the requirement pycocotools (from versions: none)
ERROR: No matching distribution found for pycocotools


In [6]:
from ultralytics import YOLO
import os, json, random, shutil
import cv2
import matplotlib.pyplot as plt
from pathlib import Path

In [2]:
import torch
print("🚀 Configuration pour NVIDIA RTX 3050...")
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

if torch.cuda.is_available():
    print(f"✅ GPU détecté : {torch.cuda.get_device_name(0)}")
    print(f"💾 VRAM disponible : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

    # Optimisations pour RTX 3050 (4GB VRAM)
    torch.backends.cudnn.benchmark = True  # Accélération cuDNN
    torch.cuda.empty_cache()  # Libérer la mémoire GPU
else:
    print("⚠️ Aucun GPU détecté, utilisation du CPU")

🚀 Configuration pour NVIDIA RTX 3050...
✅ GPU détecté : NVIDIA GeForce RTX 3070 Ti Laptop GPU
💾 VRAM disponible : 8.59 GB


In [7]:
from ultralytics import YOLO

# Charger le modèle pré-entraîné YOLO11 (nano = rapide)
model = YOLO("yolo11n.pt")

# Entraîner le modèle
results = model.train(
    data="data.yaml",
    epochs=50,
    imgsz=640,
    batch=4,
    name="sard2_yolo11",
    project="runs/train",
    workers=0,
)


Ultralytics 8.3.220  Python-3.11.13 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3070 Ti Laptop GPU, 8192MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=sard2_yolo114, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plo

KeyboardInterrupt: 

In [ ]:
from pathlib import Path

# Check all required directories
dirs_to_check = [
    'dataset/images/train',
    'dataset/images/train_rotated',
    'dataset/labels/train',
    'dataset/labels/train_rotated',
    'dataset/images/valid',
    'dataset/labels/valid',
    'dataset/images/test',
    'dataset/labels/test'
]

print("Checking dataset structure:")
all_ok = True
for dir_path in dirs_to_check:
    path = Path(dir_path)
    status = '✅' if path.exists() else '❌'
    print(f"{status} {dir_path}")
    if not path.exists():
        all_ok = False

if all_ok:
    print("\nAll directories are present. Ready to train!")
else:
    print("\n⚠️ Some directories are missing. Please check the paths above.")

In [ ]:
# Train on both original and rotated datasets
from ultralytics import YOLO

# Load the model
model = YOLO('yolo11n.pt')  # Load YOLO model

# Train the model using both datasets
results = model.train(
    data='data.yaml',        # Path to data.yaml file
    epochs=50,               # Number of epochs
    imgsz=640,              # Image size
    batch=4,                # Batch size
    name='sard2_combined',  # Run name
    project='runs/train',   # Project name
    workers=0,              # Number of worker threads
    exist_ok=True          # Overwrite existing files
)

In [ ]:
from ultralytics import YOLO
import cv2
import random
import glob
import matplotlib.pyplot as plt

# Charger ton modèle entraîné
model = YOLO("runs/train/sard2_yolo114/weights/best.pt")

# Sélectionner une image aléatoire
img_paths = glob.glob("./dataset/images/test/*.jpg")
img_path = random.choice(img_paths)
print(f"Image test: {img_path}")

# Faire l'inférence
results = model(img_path, conf=0.25)

# Annoter l'image (YOLO renvoie une image avec boxes et labels)
annotated_img = results[0].plot()

# Convertir BGR -> RGB pour affichage
annotated_img = cv2.cvtColor(annotated_img, cv2.COLOR_BGR2RGB)

# Afficher l'image avec Matplotlib
plt.figure(figsize=(10, 8))
plt.imshow(annotated_img)
plt.axis("off")
plt.title("Détections sur image test")
plt.show()

# Sauvegarder le résultat
cv2.imwrite("annotated_test.jpg", cv2.cvtColor(annotated_img, cv2.COLOR_RGB2BGR))
print("Image annotée sauvegardée -> annotated_test.jpg")

Image test: ./dataset/images/test\gss944_jpg.rf.2dc16b1f9e730587e8090cb3e5182a63.jpg

image 1/1 c:\Users\robin\OneDrive - De Vinci\Documents\A4\Machine learning\Project\projet-machine-learning\dataset\images\test\gss944_jpg.rf.2dc16b1f9e730587e8090cb3e5182a63.jpg: 384x640 (no detections), 27.1ms
Speed: 2.6ms preprocess, 27.1ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)


<Figure size 1000x800 with 1 Axes>

Image annotée sauvegardée -> annotated_test.jpg


In [ ]:
from ultralytics import YOLO
import cv2
import random
import glob
import matplotlib.pyplot as plt

# Charger ton modèle entraîné
model = YOLO("runs/train/sard2_yolo11_augmented/weights/best.pt")

# Sélectionner une image aléatoire parmi les images rotées
img_paths = glob.glob("./dataset/images/train_rotated/*.jpg")
img_path = random.choice(img_paths)
print(f"Image rotée sélectionnée: {img_path}")

# Faire l'inférence
results = model(img_path, conf=0.25)

# Annoter l'image (YOLO renvoie une image avec boxes et labels)
annotated_img = results[0].plot()

# Sauvegarder le résultat (l'image est déjà en BGR depuis YOLO)
cv2.imwrite("test_on_rotated.jpg", annotated_img)
print("Image annotée sauvegardée -> test_on_rotated.jpg")

# Convertir en RGB uniquement pour l'affichage matplotlib
annotated_img_rgb = cv2.cvtColor(annotated_img, cv2.COLOR_BGR2RGB)

# Afficher l'image originale
img_orig = cv2.imread(img_path)
img_orig_rgb = cv2.cvtColor(img_orig, cv2.COLOR_BGR2RGB)

# Créer une figure avec 2 sous-plots côte à côte
plt.figure(figsize=(20, 8))

# Image originale rotée
plt.subplot(1, 2, 1)
plt.imshow(img_orig_rgb)
plt.axis("off")
plt.title("Image rotée originale")

# Image avec détections
plt.subplot(1, 2, 2)
plt.imshow(annotated_img_rgb)
plt.axis("off")
plt.title("Détections sur image rotée")

plt.tight_layout()
plt.show()

# Afficher le nombre de détections
print(f"Nombre de détections: {len(results[0].boxes)}")
for box in results[0].boxes:
    print(f"Confiance: {box.conf.item():.2f}")

Image rotée sélectionnée: ./dataset/images/train_rotated\gss1788_jpg.rf.3087562ca209583c2fb71e192e541ce7_rot10.907038675033295.jpg

image 1/1 c:\Users\robin\OneDrive - De Vinci\Documents\A4\Machine learning\Project\projet-machine-learning\dataset\images\train_rotated\gss1788_jpg.rf.3087562ca209583c2fb71e192e541ce7_rot10.907038675033295.jpg: 384x640 (no detections), 26.6ms
Speed: 3.1ms preprocess, 26.6ms inference, 0.6ms postprocess per image at shape (1, 3, 384, 640)
Image annotée sauvegardée -> test_on_rotated.jpg


<Figure size 2000x800 with 2 Axes>

Nombre de détections: 0


In [ ]:
from ultralytics import YOLO

metrics = model.val()
print(metrics)

Ultralytics 8.3.218  Python-3.11.13 torch-2.9.0+cpu CPU (12th Gen Intel Core i7-12700H)
val: Fast image access  (ping: 0.10.0 ms, read: 1231.1864.5 MB/s, size: 664.5 KB)
val: Fast image access  (ping: 0.10.0 ms, read: 1231.1864.5 MB/s, size: 664.5 KB)
val: Scanning C:\Users\robin\OneDrive - De Vinci\Documents\A4\Machine learning\Project\projet-machine-learning\dataset\labels\valid.cache... 396 images, 2 backgrounds, 393 corrupt: 100% ━━━━━━━━━━━━ 396/396 396.5Kit/s 0.0s
val: C:\Users\robin\OneDrive - De Vinci\Documents\A4\Machine learning\Project\projet-machine-learning\dataset\images\valid\gss1015_jpg.rf.fe10e86611924a93a0ab311a93dc8c27.jpg: ignoring corrupt image/label: Label class 5 exceeds dataset class count 1. Possible class labels are 0-0
val: C:\Users\robin\OneDrive - De Vinci\Documents\A4\Machine learning\Project\projet-machine-learning\dataset\images\valid\gss1019_jpg.rf.d616261d774cd4bf253bc5aba0979bbd.jpg: ignoring corrupt image/label: Label class 5 exceeds dataset class co